<a href="https://colab.research.google.com/github/VipulBagde/DevopsProject/blob/main/_LLAVA_with_AMSS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!nvidia-smi

Wed Jul  1 18:07:28 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   47C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
!git clone https://github.com/PKU-YuanGroup/Video-LLaVA.git
%cd Video-LLaVA

Cloning into 'Video-LLaVA'...
remote: Enumerating objects: 973, done.
remote: Counting objects: 100% (504/504), done.
remote: Compressing objects: 100% (198/198), done.
remote: Total 973 (delta 414), reused 306 (delta 306), pack-reused 469 (from 1)
Receiving objects: 100% (973/973), 115.33 MiB | 18.72 MiB/s, done.
Resolving deltas: 100% (535/535), done.
/content/Video-LLaVA


In [6]:
%cd /content/Video-LLaVA
!ls

/content/Video-LLaVA
assets	 pyproject.toml  scripts		videollava
LICENSE  README.md	 TRAIN_AND_VALIDATE.md


In [1]:
!find . -maxdepth 2 -type d

.
./.config
./.config/logs
./.config/configurations
./sample_data


In [7]:
!pip install -U pip

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 63.5 MB/s eta 0:00:00
  Attempting uninstall: pip
    Found existing installation: pip 24.1.2
    Uninstalling pip-24.1.2:
      Successfully uninstalled pip-24.1.2


In [11]:
%cd /content
!rm -rf Video-LLaVA

/content


In [12]:
!pip install -U transformers accelerate sentencepiece decord av

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.6/13.6 MB 135.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.4/35.4 MB 76.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [av]


In [13]:
import transformers
import torch

print(transformers.__version__)
print(torch.__version__)

5.12.1
2.11.0+cu128


In [14]:
!nvidia-smi


Wed Jul  1 18:12:16 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   51C    P8             14W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [15]:
from transformers import VideoLlavaProcessor, VideoLlavaForConditionalGeneration

model_id = "LanguageBind/Video-LLaVA-7B-hf"

processor = VideoLlavaProcessor.from_pretrained(model_id)

model = VideoLlavaForConditionalGeneration.from_pretrained(
    model_id,
    torch_dtype="auto",
    device_map="auto"
)

print("Model Loaded Successfully!")

preprocessor_config.json:   0%|          | 0.00/890 [00:00<?, ?B/s]

[transformers] Requested torchvision backend is not available. Falling back to pil backend.


config.json:   0%|          | 0.00/998 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.59k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

...

model.safetensors.index.json:   0%|          | 0.00/112k [00:00<?, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/1077 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/148 [00:00<?, ?B/s]

Model Loaded Successfully!


In [16]:
print(model)

VideoLlavaForConditionalGeneration(
  (model): VideoLlavaModel(
    (video_tower): CLIPVisionModel(
      (embeddings): CLIPVisionEmbeddings(
        (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
        (position_embedding): Embedding(257, 1024)
      )
      (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (encoder): CLIPEncoder(
        (layers): ModuleList(
          (0-23): 24 x CLIPEncoderLayer(
            (self_attn): CLIPAttention(
              (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
              (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
            )
            (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
            (mlp): CLIPMLP(
              (activation_fn): QuickGELUActiv

In [18]:
import inspect

print(inspect.getsource(model.model.get_video_features))

    @merge_with_config_defaults
    @can_return_tuple
    @auto_docstring(
        custom_intro="Obtains video last hidden states from the vision tower and apply multimodal projection."
    )
    def get_video_features(
        self,
        pixel_values_videos: torch.FloatTensor,
        vision_feature_layer: int | list[int] | list[int] | None = None,
        output_hidden_states: bool | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple | BaseModelOutputWithPooling:
        r"""
        pixel_values_videos (`torch.FloatTensor]` of shape `(batch_size, num_frames, channels, height, width)`)
            The tensors corresponding to the input videos.
        vision_feature_layer (`Union[int, list[int]]`, *optional*):
            The index of the layer to select the vision feature. If multiple indices are provided,
            the vision feature of the corresponding indices will be concatenated to form the
            vision features.
        """
        batch_size_

In [19]:
import torch

# Create a hook on the multimodal projector
def hook(module, inp, out):
    print("=" * 60)
    print("Projector input shape :", inp[0].shape)
    print("Projector output shape:", out.shape)
    print("=" * 60)

handle = model.model.multi_modal_projector.register_forward_hook(hook)

print("Hook registered.")

Hook registered.


In [21]:
import torch
from transformers import VideoLlavaProcessor, VideoLlavaForConditionalGeneration

video_path = "VIDEO_PATH"

conversation = [
    {
        "role": "user",
        "content": [
            {"type": "video", "path": video_path},
            {"type": "text", "text": "Describe this video."},
        ],
    }
]

inputs = processor.apply_chat_template(
    conversation,
    add_generation_prompt=True,
    tokenize=True,
    return_dict=True,
    return_tensors="pt",
)

inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=50)

print(processor.batch_decode(output, skip_special_tokens=True))

ValueError: Cannot use apply_chat_template because this processor does not have a chat template.

In [22]:
print(processor)

VideoLlavaProcessor:
- image_processor: VideoLlavaImageProcessor {
  "crop_size": {
    "height": 224,
    "width": 224
  },
  "do_center_crop": true,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_processor_type": "VideoLlavaImageProcessor",
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "size": {
    "shortest_edge": 224
  }
}

- video_processor: VideoLlavaVideoProcessor {
  "crop_size": {
    "height": 224,
    "width": 224
  },
  "do_center_crop": true,
  "do_convert_rgb": true,
  "do_normalize": true,
  "do_rescale": true,
  "do_resize": true,
  "do_sample_frames": false,
  "image_mean": [
    0.48145466,
    0.4578275,
    0.40821073
  ],
  "image_std": [
    0.26862954,
    0.26130258,
    0.27577711
  ],
  "resample": 3,
  "rescale_factor": 0.00392156862745098,
  "return_meta

In [23]:
print(type(processor))

<class 'transformers.models.video_llava.processing_video_llava.VideoLlavaProcessor'>


In [24]:
print(dir(processor))

['__annotations__', '__call__', '__class__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__str__', '__subclasshook__', '__weakref__', '_auto_class', '_check_special_mm_tokens', '_get_arguments_from_pretrained', '_get_files_timestamps', '_load_tokenizer_from_pretrained', '_merge_kwargs', '_process_audio', '_process_images', '_process_videos', '_upload_modified_files', 'all_special_multimodal_tokens', 'apply_chat_template', 'audio_token_ids', 'batch_decode', 'chat_template', 'check_argument_for_proper_class', 'create_mm_token_type_ids', 'decode', 'from_args_and_dict', 'from_pretrained', 'get_attributes', 'get_possibly_dynamic_module', 'get_processor_dict', 'get_text_with_replacements', 'image_processor', 'image_token', 'image_token_id

In [25]:
import inspect

print(inspect.signature(processor.__call__))

(text: str | list[str] | list[list[str]] = None, images: Union[ForwardRef('PIL.Image.Image'), numpy.ndarray, ForwardRef('torch.Tensor'), list['PIL.Image.Image'], list[numpy.ndarray], list['torch.Tensor'], NoneType] = None, videos: Union[ForwardRef('PIL.Image.Image'), numpy.ndarray, ForwardRef('torch.Tensor'), list['PIL.Image.Image'], list[numpy.ndarray], list['torch.Tensor'], NoneType] = None, padding: bool | str | transformers.utils.generic.PaddingStrategy = False, truncation: bool | str | transformers.tokenization_utils_base.TruncationStrategy | None = None, max_length: int | None = None, return_tensors: str | transformers.utils.generic.TensorType | None = <TensorType.PYTORCH: 'pt'>) -> transformers.feature_extraction_utils.BatchFeature


In [27]:
import inspect
print(inspect.signature(processor.__call__))

(text: str | list[str] | list[list[str]] = None, images: Union[ForwardRef('PIL.Image.Image'), numpy.ndarray, ForwardRef('torch.Tensor'), list['PIL.Image.Image'], list[numpy.ndarray], list['torch.Tensor'], NoneType] = None, videos: Union[ForwardRef('PIL.Image.Image'), numpy.ndarray, ForwardRef('torch.Tensor'), list['PIL.Image.Image'], list[numpy.ndarray], list['torch.Tensor'], NoneType] = None, padding: bool | str | transformers.utils.generic.PaddingStrategy = False, truncation: bool | str | transformers.tokenization_utils_base.TruncationStrategy | None = None, max_length: int | None = None, return_tensors: str | transformers.utils.generic.TensorType | None = <TensorType.PYTORCH: 'pt'>) -> transformers.feature_extraction_utils.BatchFeature


In [28]:
import inspect
print(inspect.signature(model.model.get_video_features))

(pixel_values_videos: torch.FloatTensor, vision_feature_layer: int | list[int] | None = None, output_hidden_states: bool | None = None, **kwargs: Unpack[transformers.utils.generic.TransformersKwargs]) -> tuple | transformers.modeling_outputs.BaseModelOutputWithPooling


In [29]:
import inspect

print(inspect.getfile(model.model.__class__))

/usr/local/lib/python3.12/dist-packages/transformers/models/video_llava/modeling_video_llava.py


In [30]:
import inspect

source = inspect.getsource(model.model.get_video_features)
print(source)

    @merge_with_config_defaults
    @can_return_tuple
    @auto_docstring(
        custom_intro="Obtains video last hidden states from the vision tower and apply multimodal projection."
    )
    def get_video_features(
        self,
        pixel_values_videos: torch.FloatTensor,
        vision_feature_layer: int | list[int] | list[int] | None = None,
        output_hidden_states: bool | None = None,
        **kwargs: Unpack[TransformersKwargs],
    ) -> tuple | BaseModelOutputWithPooling:
        r"""
        pixel_values_videos (`torch.FloatTensor]` of shape `(batch_size, num_frames, channels, height, width)`)
            The tensors corresponding to the input videos.
        vision_feature_layer (`Union[int, list[int]]`, *optional*):
            The index of the layer to select the vision feature. If multiple indices are provided,
            the vision feature of the corresponding indices will be concatenated to form the
            vision features.
        """
        batch_size_

AMSS hook installed.


In [32]:
from google.colab import files

uploaded = files.upload()

video_path = list(uploaded.keys())[0]

print("Video uploaded successfully!")
print("Video path:", video_path)

Saving 8746813-uhd_4096_2160_30fps.mp4 to 8746813-uhd_4096_2160_30fps.mp4
Video uploaded successfully!
Video path: 8746813-uhd_4096_2160_30fps.mp4


In [33]:
import av
import numpy as np

container = av.open("8746813-uhd_4096_2160_30fps.mp4")

frames = []

for frame in container.decode(video=0):
    frames.append(frame.to_ndarray(format="rgb24"))

print("Total frames:", len(frames))

# Sample 8 frames uniformly
indices = np.linspace(0, len(frames)-1, 8, dtype=int)
sampled_frames = [frames[i] for i in indices]

print("Sampled:", len(sampled_frames))

Total frames: 230
Sampled: 8


In [35]:
inputs = processor.video_processor(
    sampled_frames,
    return_tensors="pt"
)

print(type(inputs))
print(inputs)

<class 'transformers.image_processing_base.BatchFeature'>
{'pixel_values_videos': tensor([[[[[ 1.7114e+00,  1.7114e+00,  1.7114e+00,  ..., -4.0541e-01,
            -3.1782e-01, -4.9300e-01],
           [ 1.7114e+00,  1.7114e+00,  1.7114e+00,  ..., -4.0451e-02,
            -2.0103e-01, -3.0322e-01],
           [ 1.7114e+00,  1.7114e+00,  1.7114e+00,  ..., -6.2439e-01,
            -5.8059e-01, -2.7403e-01],
           ...,
           [-1.2521e+00, -1.2083e+00, -1.0623e+00,  ..., -5.3680e-01,
            -4.3461e-01, -1.2804e-01],
           [-1.1207e+00, -1.1061e+00, -9.8935e-01,  ..., -1.4264e-01,
             7.6336e-02,  6.1738e-02],
           [-1.0915e+00, -1.0915e+00, -1.0185e+00,  ..., -8.4247e-02,
             1.7852e-01,  6.1738e-02]],

          [[ 1.8047e+00,  1.8047e+00,  1.8047e+00,  ..., -2.0630e-01,
            -1.0124e-01, -3.1135e-01],
           [ 1.8047e+00,  1.8047e+00,  1.8047e+00,  ...,  3.3827e-02,
            -8.6235e-02, -2.2130e-01],
           [ 1.8047e+00,  1.

In [37]:
inputs = processor.video_processor(
    sampled_frames,
    return_tensors="pt"
)

print(inputs.keys())

KeysView({'pixel_values_videos': tensor([[[[[ 1.7114e+00,  1.7114e+00,  1.7114e+00,  ..., -4.0541e-01,
            -3.1782e-01, -4.9300e-01],
           [ 1.7114e+00,  1.7114e+00,  1.7114e+00,  ..., -4.0451e-02,
            -2.0103e-01, -3.0322e-01],
           [ 1.7114e+00,  1.7114e+00,  1.7114e+00,  ..., -6.2439e-01,
            -5.8059e-01, -2.7403e-01],
           ...,
           [-1.2521e+00, -1.2083e+00, -1.0623e+00,  ..., -5.3680e-01,
            -4.3461e-01, -1.2804e-01],
           [-1.1207e+00, -1.1061e+00, -9.8935e-01,  ..., -1.4264e-01,
             7.6336e-02,  6.1738e-02],
           [-1.0915e+00, -1.0915e+00, -1.0185e+00,  ..., -8.4247e-02,
             1.7852e-01,  6.1738e-02]],

          [[ 1.8047e+00,  1.8047e+00,  1.8047e+00,  ..., -2.0630e-01,
            -1.0124e-01, -3.1135e-01],
           [ 1.8047e+00,  1.8047e+00,  1.8047e+00,  ...,  3.3827e-02,
            -8.6235e-02, -2.2130e-01],
           [ 1.8047e+00,  1.8047e+00,  1.8047e+00,  ..., -5.5148e-01,
       

In [38]:
pixel_values = inputs["pixel_values_videos"]

print(type(pixel_values))
print(pixel_values.shape)

<class 'torch.Tensor'>
torch.Size([1, 8, 3, 224, 224])


In [42]:
pixel_values = pixel_values.squeeze(0)

print(pixel_values.shape)

torch.Size([1, 8, 3, 224, 224])


In [43]:
with torch.no_grad():
    out = model.model.get_video_features(
        pixel_values_videos=pixel_values.to(model.device),
        vision_feature_layer=-2,
    )

Token embeddings BEFORE projector: torch.Size([8, 257, 1024])
Projector input shape : torch.Size([8, 257, 1024])
Projector output shape: torch.Size([8, 257, 4096])
Projector input shape : torch.Size([8, 257, 1024])
Projector output shape: torch.Size([8, 257, 4096])
After projector: torch.Size([8, 257, 4096])


amss code

In [64]:
import torch
import torch.nn as nn

class AMSS(nn.Module):

    def __init__(self, keep_ratio=0.3):
        super().__init__()
        self.keep_ratio = keep_ratio

    def forward(self, x):
        """
        x = (frames, tokens, hidden)
        """

        # motion between consecutive frames
        motion = torch.norm(
            x[1:] - x[:-1],
            dim=-1
        )

        # average motion score
        score = motion.mean(dim=0)

        # Ignore CLS token
        score[0] = score.max()

        num_keep = int(self.keep_ratio * score.shape[0])

        keep_idx = torch.topk(score, num_keep).indices

        # mask = torch.zeros_like(score)

        # mask[keep_idx] = 1.0

        # x = x * mask.unsqueeze(0).unsqueeze(-1)
        # Keep only selected tokens
        x = x[:, keep_idx, :]

        print("=" * 60)
        print("Shape after TRUE pruning:", x.shape)
        print("=" * 60)

        return x



In [66]:
amss = AMSS(keep_ratio=0.30)

In [46]:
print(amss_get_video_features)

<function amss_get_video_features at 0x7c22f5c0ae80>


In [62]:
import types
import torch

def amss_get_video_features(
    self,
    pixel_values_videos,
    vision_feature_layer=None,
    output_hidden_states=None,
    **kwargs,
):

    batch_size_vid, num_frames, channels, height, width = pixel_values_videos.shape

    pixel_values = pixel_values_videos.reshape(
        batch_size_vid * num_frames,
        channels,
        height,
        width,
    )
    kwargs["output_hidden_states"] = True
    kwargs["return_dict"] = True

    video_outputs = self.video_tower(
        pixel_values,
        **kwargs,
    )

    # Get token embeddings
    if isinstance(vision_feature_layer, int):
        video_features = video_outputs.hidden_states[vision_feature_layer]
    else:
        hs_pool = [video_outputs.hidden_states[layer] for layer in vision_feature_layer]
        video_features = torch.cat(hs_pool, dim=-1)

    print("=" * 60)
    print("Token embeddings BEFORE AMSS:", video_features.shape)

    # ==========================
    # APPLY AMSS HERE
    # ==========================
    video_features = amss(video_features)

    print("Token embeddings AFTER AMSS :", video_features.shape)
    print("=" * 60)

    # Project to LLM dimension
    video_features = self.multi_modal_projector(video_features)

    print("=" * 60)
    print("After projector:", video_features.shape)
    print("=" * 60)

    video_outputs.pooler_output = video_features

    return video_outputs

In [48]:
model.model.get_video_features = types.MethodType(
    amss_get_video_features,
    model.model,
)

print("AMSS Integrated Successfully!")

AMSS Integrated Successfully!


In [49]:
amss = AMSS(keep_ratio=0.30)

In [60]:
with torch.no_grad():
    out = model.model.get_video_features(
        pixel_values_videos=pixel_values.to(model.device),
        vision_feature_layer=-2,
    )

Token embeddings BEFORE AMSS: torch.Size([8, 257, 1024])
Shape after TRUE pruning: torch.Size([8, 77, 1024])
Token embeddings AFTER AMSS : torch.Size([8, 77, 1024])
Projector input shape : torch.Size([8, 77, 1024])
Projector output shape: torch.Size([8, 77, 4096])
Projector input shape : torch.Size([8, 77, 1024])
Projector output shape: torch.Size([8, 77, 4096])
After projector: torch.Size([8, 77, 4096])


In [67]:
import torch

prompt = "USER: <video>\nDescribe this video.\nASSISTANT:"

inputs = processor(
    text=prompt,
    videos=sampled_frames,
    return_tensors="pt",
)

inputs = {k: v.to(model.device) if torch.is_tensor(v) else v for k, v in inputs.items()}

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=50,
        do_sample=False,
    )

print(processor.decode(output[0], skip_special_tokens=True))

TypeError: CLIPVisionModel(
  (embeddings): CLIPVisionEmbeddings(
    (patch_embedding): Conv2d(3, 1024, kernel_size=(14, 14), stride=(14, 14), bias=False)
    (position_embedding): Embedding(257, 1024)
  )
  (pre_layrnorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
  (encoder): CLIPEncoder(
    (layers): ModuleList(
      (0-23): 24 x CLIPEncoderLayer(
        (self_attn): CLIPAttention(
          (k_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (v_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (q_proj): Linear(in_features=1024, out_features=1024, bias=True)
          (out_proj): Linear(in_features=1024, out_features=1024, bias=True)
        )
        (layer_norm1): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
        (mlp): CLIPMLP(
          (activation_fn): QuickGELUActivation()
          (fc1): Linear(in_features=1024, out_features=4096, bias=True)
          (fc2): Linear(in_features=4096, out_features=1024,

just checing the place


In [68]:
import inspect

print(inspect.getsource(model.model.get_video_features))

def amss_get_video_features(
    self,
    pixel_values_videos,
    vision_feature_layer=None,
    output_hidden_states=None,
    **kwargs,
):

    batch_size_vid, num_frames, channels, height, width = pixel_values_videos.shape

    pixel_values = pixel_values_videos.reshape(
        batch_size_vid * num_frames,
        channels,
        height,
        width,
    )

    video_outputs = self.video_tower(
        pixel_values,
        output_hidden_states=True,
        return_dict=True,
        **kwargs,
    )

    # Get token embeddings
    if isinstance(vision_feature_layer, int):
        video_features = video_outputs.hidden_states[vision_feature_layer]
    else:
        hs_pool = [video_outputs.hidden_states[layer] for layer in vision_feature_layer]
        video_features = torch.cat(hs_pool, dim=-1)

    print("=" * 60)
    print("Token embeddings BEFORE AMSS:", video_features.shape)

    # ==========================
    # APPLY AMSS HERE
    # ==========================
    video_

In [69]:
import types

def amss_get_video_features(
    self,
    pixel_values_videos,
    vision_feature_layer=None,
    output_hidden_states=None,
    **kwargs,
):

    batch_size_vid, num_frames, channels, height, width = pixel_values_videos.shape

    pixel_values = pixel_values_videos.reshape(
        batch_size_vid * num_frames,
        channels,
        height,
        width,
    )

    # Configure kwargs only once
    kwargs["output_hidden_states"] = True
    kwargs["return_dict"] = True

    video_outputs = self.video_tower(
        pixel_values,
        **kwargs,
    )

    if isinstance(vision_feature_layer, int):
        video_features = video_outputs.hidden_states[vision_feature_layer]
    else:
        hs_pool = [video_outputs.hidden_states[layer] for layer in vision_feature_layer]
        video_features = torch.cat(hs_pool, dim=-1)

    print("Before:", video_features.shape)

    video_features = amss(video_features)

    print("After:", video_features.shape)

    video_features = self.multi_modal_projector(video_features)

    video_outputs.pooler_output = video_features

    return video_outputs

model.model.get_video_features = types.MethodType(
    amss_get_video_features,
    model.model,
)

print("NEW FUNCTION INSTALLED")

NEW FUNCTION INSTALLED


In [ ]:
checkking placeholder code

In [79]:
import inspect

print(inspect.getsource(type(processor).__call__))

    @auto_docstring
    def __call__(
        self,
        text: TextInput | PreTokenizedInput | list[TextInput] | list[PreTokenizedInput] = None,
        images: ImageInput | None = None,
        videos: ImageInput | None = None,
        padding: bool | str | PaddingStrategy = False,
        truncation: bool | str | TruncationStrategy | None = None,
        max_length: int | None = None,
        return_tensors: str | TensorType | None = TensorType.PYTORCH,
    ) -> BatchFeature:
        r"""
        padding (`bool`, `str` or [`~utils.PaddingStrategy`], *optional*, defaults to `False`):
            Select a strategy to pad the returned sequences (according to the model's padding side and padding
            index) among:
            - `True` or `'longest'`: Pad to the longest sequence in the batch (or no padding if only a single
                sequence if provided).
            - `'max_length'`: Pad to a maximum length specified with the argument `max_length` or to the maximum
      

In [80]:
import types

original_processor_call = processor.__class__.__call__

print("Original processor backed up.")

Original processor backed up.


In [82]:
import numpy as np
from transformers.feature_extraction_utils import BatchFeature
from transformers.image_utils import get_image_size, to_numpy_array

def patched_processor_call(
    self,
    text=None,
    images=None,
    videos=None,
    padding=False,
    truncation=None,
    max_length=None,
    return_tensors="pt",
):

    if isinstance(text, str):
        text = [text]

    data = {}

    if videos is not None:

        encoded_videos = self.video_processor(
            videos=videos,
            return_tensors=return_tensors,
        )

        data.update(encoded_videos)

        one_video = encoded_videos["pixel_values_videos"][0]

        if isinstance(one_video, (list, tuple)):
            one_video = np.array(one_video)
        else:
            one_video = to_numpy_array(one_video)

        height, width = get_image_size(one_video[0])
        num_frames = one_video.shape[0]

        num_image_tokens = (
            (height // self.patch_size)
            * (width // self.patch_size)
        )

        num_image_tokens += self.num_additional_image_tokens

        # ---------- OUR CHANGE ----------
        kept_tokens = int(num_image_tokens * amss.keep_ratio)

        num_video_tokens = kept_tokens * num_frames

        print("Original/frame :", num_image_tokens)
        print("Kept/frame     :", kept_tokens)
        print("Total video tokens:", num_video_tokens)
        # -------------------------------

        text = [
            sample.replace(
                self.video_token,
                self.video_token * num_video_tokens,
            )
            for sample in text
        ]

    text_inputs = self.tokenizer(
        text,
        return_tensors=None,
        padding=padding,
        truncation=truncation,
        max_length=max_length,
    )

    data.update(text_inputs)

    return BatchFeature(
        data=data,
        tensor_type=return_tensors,
    )

In [83]:
processor.__class__.__call__ = patched_processor_call

print("Dynamic processor installed.")

Dynamic processor installed.


In [84]:
prompt = "USER: <video>\nDescribe this video.\nASSISTANT:"

inputs = processor(
    text=prompt,
    videos=sampled_frames,
    return_tensors="pt",
)

video_token_id = model.config.video_token_index

print("Input shape:", inputs["input_ids"].shape)
print(
    "Video token count:",
    (inputs["input_ids"] == video_token_id).sum().item()
)

Original/frame : 257
Kept/frame     : 77
Total video tokens: 616
Input shape: torch.Size([1, 632])
Video token count: 616


In [90]:
import json

results = {
    "Model": "Video-LLaVA + AMSS",
    "Original Tokens": 2056,
    "Remaining Tokens": 616,
    "Reduction (%)": round((1-616/2056)*100,2),
    "Inference Time (s)": round(end-start,3),
    "Peak GPU Memory (MB)": round(peak_gpu,2),
    "CPU RAM Used (GB)": round(ram.used/1024**3,2),
    "Output": processor.decode(output[0], skip_special_tokens=True)
}

with open("amss_results.json","w") as f:
    json.dump(results,f,indent=4)

print(results)

{'Model': 'Video-LLaVA + AMSS', 'Original Tokens': 2056, 'Remaining Tokens': 616, 'Reduction (%)': 70.04, 'Inference Time (s)': 311.288, 'Peak GPU Memory (MB)': 13377.44, 'CPU RAM Used (GB)': 11.5, 'Output': "USER: \nDescribe this video.\nASSISTANT: The video features a large white dog standing on a lush green lawn. The dog appears to be enjoying the outdoors and is looking up at the camera. The dog's fur is well-groomed, and it appears"}


In [91]:
import inspect

with open("AMSS_patch.py", "w") as f:

    f.write("# ==============================\n")
    f.write("# AMSS CLASS\n")
    f.write("# ==============================\n\n")
    f.write(inspect.getsource(AMSS))

    f.write("\n\n")

    f.write("# ==============================\n")
    f.write("# get_video_features PATCH\n")
    f.write("# ==============================\n\n")
    f.write(inspect.getsource(amss_get_video_features))

    f.write("\n\n")

    f.write("# ==============================\n")
    f.write("# Processor PATCH\n")
    f.write("# ==============================\n\n")
    f.write(inspect.getsource(patched_processor_call))

print("Saved AMSS_patch.py")

OSError: source code not available